# 02 ASVspoof 2019 LA audit
Can the label be predicted without listening to the speech?

In [ ]:
import warnings
import pandas as pd
from audio_df.data.manifests import load_manifest, summarise_manifest, assert_no_speaker_overlap, stratified_subset
from audio_df.helpers.eval_audio import trimmed_manifest
from audio_df.helpers.research import leakage

m = {s: load_manifest(f"../data/manifests/{s}.csv", "..") for s in ["train", "dev", "eval"]}

## Splits

In [ ]:
summarise_manifest(pd.concat(m.values()))

In [ ]:
assert_no_speaker_overlap(m["train"], m["dev"])
print("train/dev: no shared speakers")

Train/dev only have A01–A06, eval only A07–A19, so eval = unseen attacks. About 9 spoof per bona fide in every split, so we report EER, not accuracy.

## Leakage: raw vs trimmed
Tiny classifiers on non-speech features only (2,000 dev clips). AUC 0.5 = no shortcut, > 0.65 = flagged (project convention).

In [ ]:
raw = stratified_subset(m["dev"], 2000)
trim = trimmed_manifest(m["dev"], "../data/cache", "dev").set_index("utt_id").loc[raw.utt_id].reset_index()

def audit(df):
    feats = leakage.extract_nuisance_features(df)[["rms", "silence_leading", "silence_trailing", "silence_total"]]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return leakage.audit_all(feats, df.label)[["feature", "auc", "eer"]]

audit(raw).merge(audit(trim), on="feature", suffixes=("_raw", "_trimmed")).round(3)

Leading silence alone predicts the label (AUC 0.77) on raw audio. After trimming every feature is ~0.5, so trimming removes the shortcut.